# 📊 NeuralShield v6 — Validation-1: Küme-Bootstrap CI + Per-Dil AUC + Recall@FP=1%

**Amaç:** v6.2 ile diğerleri/Patronus arasındaki farkların **gürültü mü gerçek mi** olduğunu göstermek. Yeni eğitim YOK.

- **Birincil:** Recall@FP=1% (eşikten bağımsız), AUC, Macro-F1@0.7 — **%95 bootstrap CI** ile.
- **Cluster-bootstrap:** aynı cümlenin 12 çevirisi bağımlı → **dil-küme bootstrap** (12 dil yeniden örneklenir) + **per-dil satır-bootstrap**.
- Çıktı: tablo + `validation_ci_v6.json`.

In [ ]:
# @title 1. Kurulum + Drive + Repo
import os, sys, json, re, random, gc, time
import warnings; warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM']='false'
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e: print('drive:', e)
!pip -q install -U transformers datasets scikit-learn 2>/dev/null
if not os.path.exists('ai-soc-agent-lab'):
    !git clone -q https://github.com/Mustafa-Caliskan/ai-soc-agent-lab.git
if os.path.exists('ai-soc-agent-lab'): os.chdir('ai-soc-agent-lab')
!git pull -q
import numpy as np, torch, pandas as pd
from sklearn.metrics import roc_auc_score, f1_score
MASTER='/content/drive/MyDrive/NeuralShield_v6_MasterData'
FAIR=MASTER+'/heldout_benchmark_v6_fair.json'
data=json.load(open(FAIR,encoding='utf-8'))
texts=[x['text'] for x in data]; labels=np.array([int(x['label']) for x in data])
langs=np.array([x['language'] for x in data])
print('N=',len(data),'| dil:',dict(pd.Series(langs).value_counts()))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# @title 2. Modelleri yükle + olasılıkları hesapla
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
def best_ckpt(base):
    if os.path.exists(base+'/config.json'): return base
    ck=base+'/checkpoints'
    if os.path.isdir(ck):
        subs=[d for d in os.listdir(ck) if d.startswith('checkpoint-')]
        if subs: return os.path.join(ck, sorted(subs,key=lambda x:int(x.split('-')[1]))[-1])
    return None
def eval_ours(path,bs=64):
    tok=AutoTokenizer.from_pretrained(path); mod=AutoModelForSequenceClassification.from_pretrained(path,torch_dtype=torch.float16).to('cuda').eval()
    pr=[]
    for i in range(0,len(texts),bs):
        inp=tok(texts[i:i+bs],return_tensors='pt',padding=True,truncation=True,max_length=512).to('cuda')
        with torch.no_grad(): pr.extend(torch.softmax(mod(**inp).logits.float(),-1)[:,1].cpu().numpy().tolist())
    del mod,tok; gc.collect(); torch.cuda.empty_cache(); return np.array(pr)
def eval_rival(mid,bs=64):
    clf=pipeline('text-classification',model=mid,truncation=True,max_length=2048,device=0,torch_dtype=torch.float16)
    res=clf(texts,batch_size=bs)
    pos=('INJECTION','LABEL_1','ATTACK','UNSAFE','JAILBREAK','MALICIOUS')
    pr=np.array([r['score'] if any(p in str(r['label']).upper() for p in pos) else 1-r['score'] for r in res])
    del clf; gc.collect(); torch.cuda.empty_cache(); return pr
PROBS={}
CAND=[('mmBERT v6.2', '/content/drive/MyDrive/NeuralShield_v6_mmbert/checkpoints_v6_2_backup'),
      ('mmBERT v6.3B', '/content/drive/MyDrive/NeuralShield_v6_mmbert_stageB'),
      ('BGE-M3 v6.2', '/content/drive/MyDrive/NeuralShield_v6_bgem3'),
      ('XLM-R v6.2', '/content/drive/MyDrive/NeuralShield_v6_xlmr')]
for name,base in CAND:
    ck=best_ckpt(base)
    if ck:
        print('yükleniyor:',name,ck); PROBS[name]=eval_ours(ck)
    else: print('[i] atlandı:',name)
try:
    print('yükleniyor: Patronus'); PROBS['Patronus']=eval_rival('patronus-studio/wolf-defender-prompt-injection')
except Exception as e: print('Patronus atlandı:',str(e)[:80])
json.dump({k:v.tolist() for k,v in PROBS.items()}, open(MASTER+'/val_probs.json','w'))
print('modeller:', list(PROBS.keys()))

In [ ]:
# @title 3. Metrik + Küme-Bootstrap CI + Per-Dil AUC
def metrics_at(p, idx):
    y=labels[idx]; pr=p[idx]
    if len(np.unique(y))<2: return None
    auc=roc_auc_score(y,pr)
    f1=f1_score(y,(pr>=0.7).astype(int),average='macro')
    ben=pr[y==0]; atk=pr[y==1]
    thr=np.quantile(ben,0.99)            # benign'in ~%1'i esik ustunde -> oracle Recall@FP=1%
    rec=(atk>=thr).mean()
    return {'auc':auc,'f1_07':f1,'rec_fp1':rec}
def boot(p, groups, B=500):
    rng=np.random.default_rng(42); vals={'auc':[],'f1_07':[],'rec_fp1':[]}
    for _ in range(B):
        sel=rng.integers(0,len(groups),len(groups))
        idx=np.concatenate([groups[k] for k in sel])
        m=metrics_at(p,idx)
        if m:
            for k in vals: vals[k].append(m[k])
    return {k:(float(np.percentile(v,2.5)),float(np.percentile(v,97.5))) for k,v in vals.items() if v}
LANGLIST=[l for l in ['en','tr','de','fr','es','it','pt','ru','ar','ja','ko','zh'] if (langs==l).any()]
lang_groups=[np.where(langs==l)[0] for l in LANGLIST]
all_idx=np.arange(len(labels))
rows=[]
for name,p in PROBS.items():
    point=metrics_at(p,all_idx)
    ci_lang=boot(p, lang_groups, B=500)   # dil-kume bootstrap (12 kume)
    rows.append({'Model':name,
                 'AUC':round(point['auc'],4),'AUC_CI':(round(ci_lang['auc'][0],4),round(ci_lang['auc'][1],4)),
                 'F1@0.7':round(point['f1_07'],4),'F1_CI':(round(ci_lang['f1_07'][0],4),round(ci_lang['f1_07'][1],4)),
                 'Recall@FP=1%':round(point['rec_fp1'],4),'Rec_CI':(round(ci_lang['rec_fp1'][0],4),round(ci_lang['rec_fp1'][1],4))})
display(pd.DataFrame(rows))
print('\n--- PER-DIL AUC (point + satir-bootstrap CI) ---')
perlang={}
for name,p in PROBS.items():
    perlang[name]={}
    for l,g in zip(LANGLIST,lang_groups):
        m=metrics_at(p,g)
        if not m: continue
        ci=boot(p,[np.array([i]) for i in g], B=300)
        perlang[name][l]={'auc':round(m['auc'],3),'auc_ci':(round(ci['auc'][0],3),round(ci['auc'][1],3)) if 'auc' in ci else None}
    print(name, '|', ', '.join(f"{l}:{perlang[name][l]['auc']}" for l in LANGLIST))
json.dump({'overall':rows,'per_language_auc':perlang}, open(MASTER+'/validation_ci_v6.json','w'), ensure_ascii=False, indent=2)
print('\n-> validation_ci_v6.json kaydedildi')